In [ ]:
!unzip -q "/content/AI_LAB_dataset.zip" -d "/content/raw_data"

In [ ]:
import os
import cv2
import numpy as np
from sklearn.model_selection import train_test_split

base_dir = "/content/raw_data/AI_LAB_dataset"
categories = ["aam_pata", "kathal_pata"]

X = []
y = []

for category in categories:
    folder_path = os.path.join(base_dir, category)
    label = 0 if category == "aam_pata" else 1

    for img_name in os.listdir(folder_path):
        img_path = os.path.join(folder_path, img_name)
        img = cv2.imread(img_path)

        if img is not None:
            img = cv2.resize(img, (128, 128))
            X.append(img)
            y.append(label)

X = np.array(X) / 255.0
y = np.array(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Total images loaded: {len(X)}")
print(f"Training data (80%): {X_train.shape[0]} images")
print(f"Testing data (20%): {X_test.shape[0]} images")
print(f"Sample labels (y_train): {y_train[:10]}")

In [ ]:
import os
import cv2
import numpy as np

output_dir = "/content/processed_dataset"

for split in ['train', 'test']:
    for label in [0, 1]:
        os.makedirs(os.path.join(output_dir, split, str(label)), exist_ok=True)

def save_data_to_folders(X_data, y_data, split_name):
    for index in range(len(X_data)):
        img = (X_data[index] * 255).astype(np.uint8)
        label = y_data[index]
        file_path = os.path.join(output_dir, split_name, str(label), f"image_{index}.png")
        cv2.imwrite(file_path, img)

print("Saving training images...")
save_data_to_folders(X_train, y_train, 'train')

print("Saving testing images...")
save_data_to_folders(X_test, y_test, 'test')

print(f"Done! Your processed data is saved at: {output_dir}")

Saving training images...
Saving testing images...
Done! Your processed data is saved at: /content/processed_dataset


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

print("--- Preparing train/validation/test splits ---")

X_train = X_train.astype("float32")
X_test = X_test.astype("float32")
y_train = y_train.astype("float32").reshape(-1)
y_test = y_test.astype("float32").reshape(-1)

if X_train.max() > 1:
    X_train = X_train / 255.0
    X_test = X_test / 255.0

X_train_mlp, X_val, y_train_mlp, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
    stratify=y_train
)

print("Training samples:", len(X_train_mlp))
print("Validation samples:", len(X_val))
print("Test samples:", len(X_test))

print("\n--- Building improved MLP model ---")

mlp_model = models.Sequential([
    layers.Input(shape=(128, 128, 3)),
    layers.Flatten(),
    layers.Dense(8, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.35),
    layers.Dense(6, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.30),
    layers.Dense(4, activation="relu"),
    layers.BatchNormalization(),
    layers.Dropout(0.25),
    layers.Dense(1, activation="sigmoid")
])

mlp_model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

mlp_model.summary()

print("\n--- Training MLP model ---")

training_callbacks = [
    callbacks.EarlyStopping(
        monitor="val_loss",
        patience=6,
        restore_best_weights=True,
        verbose=1
    ),
    callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

mlp_history = mlp_model.fit(
    X_train_mlp,
    y_train_mlp,
    epochs=50,
    batch_size=32,
    validation_data=(X_val, y_val),
    callbacks=training_callbacks,
    verbose=1
)

mlp_loss, mlp_acc = mlp_model.evaluate(
    X_test,
    y_test,
    verbose=0
)

print(f"\nMLP Final Test Loss: {mlp_loss:.4f}")
print(f"MLP Final Test Accuracy: {mlp_acc * 100:.2f}%")

history = mlp_history.history
epochs_range = range(1, len(history["accuracy"]) + 1)

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, history["accuracy"], marker="o", label="Training Accuracy")
plt.plot(epochs_range, history["val_accuracy"], marker="o", label="Validation Accuracy")
plt.title("MLP Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.grid(alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history["loss"], marker="o", label="Training Loss")
plt.plot(epochs_range, history["val_loss"], marker="o", label="Validation Loss")
plt.title("MLP Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.grid(alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

y_prob = mlp_model.predict(X_test, verbose=0).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print("\n--- Classification Report ---")
print(classification_report(y_test, y_pred, target_names=["am_pata", "kathal_pata"]))

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=["am_pata", "kathal_pata"], yticklabels=["am_pata", "kathal_pata"])
plt.title("MLP Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()

In [ ]:
class_labels = {0: "am_pata", 1: "kathal_pata"}

import matplotlib.pyplot as plt
import cv2
import numpy as np

plt.figure(figsize=(15, 12))

random_indices = np.random.choice(len(X_test), 20, replace=False)

for i, idx in enumerate(random_indices):
    plt.subplot(4, 5, i + 1)

    img_bgr = (X_test[idx] * 255).astype('uint8')
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    plt.imshow(img_rgb)

    actual_label = class_labels[int(y_test[idx])]
    predicted_label = class_labels[y_pred[idx]]

    title_color = "green" if actual_label == predicted_label else "red"

    plt.title(f"True: {actual_label}\nPred: {predicted_label}", color=title_color, fontsize=10)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

leaf_cnn = models.Sequential()

leaf_cnn.add(layers.Conv2D(8, (3, 3), activation='relu', input_shape=(128, 128, 3)))
leaf_cnn.add(layers.MaxPooling2D((2, 2)))

leaf_cnn.add(layers.Conv2D(8, (3, 3), activation='relu'))
leaf_cnn.add(layers.MaxPooling2D((2, 2)))

leaf_cnn.add(layers.Conv2D(8, (3, 3), activation='relu'))
leaf_cnn.add(layers.MaxPooling2D((2, 2)))

leaf_cnn.add(layers.Flatten())
leaf_cnn.add(layers.Dense(1, activation='sigmoid'))

leaf_cnn.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

leaf_cnn.summary()

In [ ]:
leaf_cnn.fit(
    X_train,
    y_train,
    epochs=15,
    batch_size=32,
    validation_split=0.2,
    verbose=1
)

test_loss, test_acc = leaf_cnn.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc * 100:.2f}%")

y_pred_prob = leaf_cnn.predict(X_test)
y_pred = (y_pred_prob >= 0.5).astype(int).ravel()

class_labels = {0: "aam_pata", 1: "kathal_pata"}

import matplotlib.pyplot as plt
import cv2

plt.figure(figsize=(15, 12))

for i in range(20):
    plt.subplot(4, 5, i + 1)

    img_bgr = (X_test[i] * 255).astype('uint8')
    img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)

    plt.imshow(img_rgb)

    actual_label = class_labels[int(y_test[i])]
    predicted_label = class_labels[y_pred[i]]

    title_color = "green" if actual_label == predicted_label else "red"

    plt.title(f"True: {actual_label}\nPred: {predicted_label}", color=title_color, fontsize=10)
    plt.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report

test_loss, test_acc = leaf_cnn.evaluate(X_test, y_test, verbose=0)
print(f"Test Loss: {test_loss:.4f}")
print(f"Test Accuracy: {test_acc * 100:.2f}%\n")

y_pred_prob_cnn = leaf_cnn.predict(X_test, verbose=0)
y_pred_cnn = (y_pred_prob_cnn >= 0.5).astype(int).ravel()

print("--- Classification Report ---")
print(classification_report(y_test, y_pred_cnn, target_names=["aam_pata", "kathal_pata"]))

history = leaf_cnn_history.history
epochs_range = range(1, len(history["accuracy"]) + 1)

plt.figure(figsize=(14, 5))

plt.subplot(1, 2, 1)
plt.plot(epochs_range, history["accuracy"], marker="o", label="Training Accuracy")
plt.plot(epochs_range, history["val_accuracy"], marker="o", label="Validation Accuracy")
plt.title("CNN Training vs Validation Accuracy")
plt.xlabel("Epoch")
plt.ylabel("Accuracy")
plt.grid(alpha=0.3)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs_range, history["loss"], marker="o", label="Training Loss")
plt.plot(epochs_range, history["val_loss"], marker="o", label="Validation Loss")
plt.title("CNN Training vs Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("Binary Cross-Entropy Loss")
plt.grid(alpha=0.3)
plt.legend()

plt.tight_layout()
plt.show()

cm_cnn = confusion_matrix(y_test, y_pred_cnn)

plt.figure(figsize=(6, 5))
sns.heatmap(
    cm_cnn,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["aam_pata", "kathal_pata"],
    yticklabels=["aam_pata", "kathal_pata"]
)
plt.title("CNN Confusion Matrix")
plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.show()